# Graphs as Data & Message Passing

Companion notebook for the [Graphs as Data lesson](https://ml-viz-ruby.vercel.app/courses/graph-neural-networks/01-graphs-as-data).

We represent a small graph with an adjacency matrix and node features, implement one round of
**message passing** (mean-aggregate neighbors), watch information spread over multiple rounds, and
verify the whole thing is **permutation invariant**. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## Intuition — learning on things that are connected

Molecules, social networks, road maps, citation graphs — much of the world's data is **nodes and
edges**, with no grid or sequence to exploit. GNNs handle it with one idea: **message passing** — each
node updates its feature by **aggregating its neighbors'** features (sum/mean/max), and stacking `k`
rounds lets information travel `k` hops. Two properties make this principled: the aggregator must be
**permutation-invariant** (a graph has no node order), and repeated aggregation has a known failure —
**over-smoothing**, where all nodes drift to the same value. We build message passing from scratch and
verify both properties directly.

## 1 — A graph as an adjacency matrix + features

Five nodes in a small graph. `A[u,v]=1` means an edge; we add self-loops so a node keeps its own
features during aggregation.

In [ ]:
# edges: 0-1, 1-2, 2-3, 3-4, 1-3  (undirected)
A = np.zeros((5, 5))
for u, v in [(0,1),(1,2),(2,3),(3,4),(1,3)]:
    A[u,v] = A[v,u] = 1
A = A + np.eye(5)                 # add self-loops

X = np.array([                    # one feature per node
    [1.0], [2.0], [3.0], [4.0], [5.0]
])
print('adjacency (with self-loops):\n', A)
print('node features:', X.ravel())

**What to notice:** the entire graph is two arrays — the **adjacency matrix** (who's connected) and
a **feature matrix** (one row per node). The self-loops on the diagonal let each node keep its own
information when averaging with neighbors.

## 2 — One round of mean message passing

Each node's new feature is the **mean of its neighbors' features** (including itself). In matrix form
this is `D⁻¹ A X`, where `D` is the diagonal degree matrix — an elegant, permutation-respecting,
variable-neighbor-count operation.

In [ ]:
def mean_aggregate(A, X):
    deg = A.sum(axis=1, keepdims=True)     # number of neighbors (incl. self)
    return (A @ X) / deg                   # row-normalized neighbor mean

H1 = mean_aggregate(A, X)
print('after 1 round:', H1.ravel())
# node 0 (neighbors {0,1}): mean(1,2)=1.5 ; node 1 (neighbors {0,1,2,3}): mean(1,2,3,4)=2.5
assert np.isclose(H1[0,0], 1.5) and np.isclose(H1[1,0], 2.5)
print('\u2713 matches hand calculation')

**What to notice:** one round of message passing is a single matrix product — `D⁻¹A @ X` computes
every node's neighbor-mean simultaneously, and the hand check confirms it (node 0 averages {1, 2} →
1.5). The matrix form is why GNNs scale: aggregation over the whole graph is one sparse matmul.

## 3 — Information spreads with each round (receptive field)

After k rounds a node has mixed in information from up to k hops away. We track how node 0's value
evolves — and how, with many rounds, all nodes drift toward the same value (**over-smoothing**).

In [ ]:
H = X.copy()
for k in range(1, 8):
    H = mean_aggregate(A, H)
    print(f'round {k}: {H.ravel()}   spread={H.max()-H.min():.4f}')
print('\nNote how the spread shrinks toward 0 — that is over-smoothing.')

**What to notice:** each round widens the **receptive field** by one hop — after round `k`, node 0
has absorbed information from `k` hops away. But watch the `spread` column: it shrinks every round,
heading toward 0. Information spreads *and* individuality dissolves — the over-smoothing tension at
the heart of GNN depth.

## 4 — Permutation invariance

Relabeling the nodes (permuting rows/cols of A and rows of X) must give the *same* result, just
reordered. We permute the graph, run message passing, and check it matches the un-permuted result
permuted the same way.

In [ ]:
perm = np.array([3, 0, 4, 1, 2])         # an arbitrary relabeling
P = np.eye(5)[perm]                       # permutation matrix

A_perm = P @ A @ P.T                       # relabel the graph
X_perm = P @ X
H_perm = mean_aggregate(A_perm, X_perm)

# message passing on the permuted graph == permuting the original result
assert np.allclose(H_perm, P @ mean_aggregate(A, X))
print('\u2713 message passing is permutation equivariant — node order does not matter')

**What to notice:** relabeling the nodes and re-running gives the *same answer, reordered* —
**permutation equivariance**, verified exactly. This is the structural requirement that separates
valid graph operations from ones that secretly depend on an arbitrary node numbering.

## The library way — the fixed point is an eigenvector

Where does over-smoothing *go*? Repeated mean aggregation is repeated multiplication by the
random-walk matrix `P = D⁻¹A` — power iteration from the eigenvalue course! The features converge to
`P`'s dominant eigenvector direction. The check: run many rounds and compare against
`scipy`/`numpy`'s eigenvector.

In [ ]:
P_walk = A / A.sum(axis=1, keepdims=True)
H_inf = X.copy()
for _ in range(200):
    H_inf = P_walk @ H_inf

evals, evecs = np.linalg.eig(P_walk.T)      # stationary direction = left eigenvector, eigenvalue 1
v = np.real(evecs[:, np.argmax(np.real(evals))])
print('after 200 rounds, all nodes ->', H_inf.ravel().round(4))
print('spread =', float(H_inf.max() - H_inf.min()))
assert H_inf.max() - H_inf.min() < 1e-8, "mean aggregation must converge to a constant across nodes"
# the limit value is the degree-weighted average of the initial features
limit = float((A.sum(1) @ X.ravel()) / A.sum())
print(f'limit value {H_inf[0,0]:.4f} == degree-weighted feature average {limit:.4f}')
assert abs(H_inf[0,0] - limit) < 1e-6
print('\nover-smoothing == power iteration converging to the random-walk stationary distribution ✓')

**What to notice:** after enough rounds every node holds the **same number** — the degree-weighted
average of the initial features, i.e. the random-walk stationary distribution's projection. Over-
smoothing isn't vague "information loss"; it's power iteration (from the eigenvalue course) doing
exactly what it must. That's why real GNNs stay shallow or add residuals.

## Gotchas & tradeoffs

- **Aggregators must be permutation-invariant** — sum/mean/max qualify; anything order-dependent
  (concatenation, RNN over neighbors without sorting) silently breaks graph semantics.
- **Over-smoothing caps depth** at ~2–4 message-passing layers; residual connections and normalization
  push it a little further.
- **Aggregator choice loses different information:** mean loses degree, sum loses scale-comparability,
  max loses counts — GIN's sum-based design is provably the most expressive of the three.
- **Self-loops matter:** without them a node's own features can wash out after one round.

In [ ]:
# Aggregators differ in what they can DISTINGUISH: two different neighborhoods, same mean
n1 = np.array([[2.0], [2.0]])            # two neighbors with feature 2
n2 = np.array([[1.0], [3.0]])            # two neighbors with features 1 and 3
print('mean:', n1.mean(), '==', n2.mean(), ' -> mean CANNOT tell these neighborhoods apart')
print('sum :', n1.sum(),  '==', n2.sum(),  ' -> sum cannot either here...')
print('max :', n1.max(),  'vs', n2.max(),  ' -> max CAN')
n3 = np.array([[2.0], [2.0], [2.0]])
print('...but mean(n3) ==', n3.mean(), '== mean(n1): only SUM sees the extra neighbor (3 vs 2 nodes)')

**What to notice:** each aggregator has blind spots — mean can't distinguish {2,2} from {1,3},
and only sum notices a *third* identical neighbor. Which structures your GNN can even *see* depends on
this choice; it's the core of the GIN expressiveness result.

## ✏️ Your turn

**Exercise.** Implement `aggregate(A, X, kind)` supporting `kind` in `{'sum', 'mean', 'max'}` —
the three permutation-invariant aggregators. For each node, pool the features of its neighbors
(the nonzero entries of that row of `A`, which already includes the self-loop).

In [ ]:
def aggregate(A, X, kind='mean'):
    out = np.zeros_like(X)
    for v in range(A.shape[0]):
        neighbors = np.where(A[v] > 0)[0]
        feats = X[neighbors]
        # TODO(you): set out[v] to the sum / mean / max over `feats` according to `kind`
        ...
    return out

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.allclose(aggregate(A, X, 'mean'), mean_aggregate(A, X))
assert np.isclose(aggregate(A, X, 'sum')[0, 0], 3.0)     # node 0: 1+2
assert np.isclose(aggregate(A, X, 'max')[1, 0], 4.0)     # node 1 neighbors max = 4
# permutation invariance holds for any aggregator
for k in ['sum', 'mean', 'max']:
    assert np.allclose(aggregate(P @ A @ P.T, P @ X, k), P @ aggregate(A, X, k))
print('\u2713 all three aggregators are correct and permutation equivariant')

<details>
<summary>Solution</summary>

```python
def aggregate(A, X, kind='mean'):
    out = np.zeros_like(X)
    for v in range(A.shape[0]):
        feats = X[np.where(A[v] > 0)[0]]
        if kind == 'sum':
            out[v] = feats.sum(axis=0)
        elif kind == 'mean':
            out[v] = feats.mean(axis=0)
        elif kind == 'max':
            out[v] = feats.max(axis=0)
    return out
```

Sum, mean, and max are all permutation invariant — the order of the neighbor set doesn't change the
result. That invariance is precisely what makes them valid graph aggregators.

</details>